In [3]:
"""
Attribution Entropy: Teacher vs Student interpretability comparison.

NOTE: test.parquet already contains head+tail truncated text in column "text",
so no further truncation is needed.
"""

import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers_interpret import SequenceClassificationExplainer

# ── Setup paths ──────────────────────────────────────────────────────
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
from project_paths import get_paths

paths = get_paths(ROOT)
DATA_DIR = paths.data_processed
CKPT_DIR = paths.checkpoints
device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load models ──────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

teacher_path = CKPT_DIR / "bert_teacher_finetuned" / "checkpoint-18000"
student_phase2_path = CKPT_DIR / "tinybert_phase2" / "checkpoint-13000"

teacher = AutoModelForSequenceClassification.from_pretrained(teacher_path).to(device).eval()
student = AutoModelForSequenceClassification.from_pretrained(student_phase2_path).to(device).eval()

# ── Load test data ───────────────────────────────────────────────────
test_ds = load_dataset("parquet", data_files=str(DATA_DIR / "test.parquet"))["train"]

# ── Sample reviews (balanced) ────────────────────────────────────────
N_PER_CLASS = 100  # 200 total
spoiler_idx = [i for i, lab in enumerate(test_ds["label"]) if lab == 1]
nospoiler_idx = [i for i, lab in enumerate(test_ds["label"]) if lab == 0]
rng = np.random.default_rng(42)
sample_idx = np.concatenate([
    rng.choice(spoiler_idx, size=min(N_PER_CLASS, len(spoiler_idx)), replace=False),
    rng.choice(nospoiler_idx, size=min(N_PER_CLASS, len(nospoiler_idx)), replace=False),
])
texts = [test_ds[int(i)]["text"] for i in sample_idx]
labels = [test_ds[int(i)]["label"] for i in sample_idx]

# ── Sanity check ─────────────────────────────────────────────────────
print("=" * 70)
print("SANITY CHECK: verifying token counts from parquet")
print("=" * 70)
max_found = 0
for i, txt in enumerate(texts):
    n_tok = len(tokenizer.encode(txt, add_special_tokens=True))
    max_found = max(max_found, n_tok)
    if i < 5:
        label_str = "SPOILER" if labels[i] == 1 else "NO-SPOI"
        print(f"  [{i}] {label_str} | {n_tok:3d} tokens | \"{txt[:80]}...\"")
    if n_tok > 512:
        print(f"  WARNING: sample {i} has {n_tok} tokens!")
print(f"  Max tokens: {max_found} (limit: 512)")
print(f"  Total: {len(texts)} ({sum(labels)} spoiler, {len(labels)-sum(labels)} no-spoiler)\n")


# ── Attribution entropy ──────────────────────────────────────────────
def attribution_entropy(explainer, text):
    try:
        word_attributions = explainer(text)
    except Exception as e:
        print(f"  ERROR: {e}")
        return None
    scores = np.array([abs(s) for _, s in word_attributions])
    scores = scores[1:-1]  # strip [CLS] and [SEP]
    if scores.sum() == 0:
        return 0.0
    p = scores / scores.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


teacher_explainer = SequenceClassificationExplainer(teacher, tokenizer)
student_explainer = SequenceClassificationExplainer(student, tokenizer)

teacher_entropies = []
student_entropies = []

print("Computing attributions...")
for i, txt in enumerate(texts):
    label_str = "SPO" if labels[i] == 1 else "NO "
    print(f"  [{i+1:3d}/{len(texts)}] {label_str} ", end="", flush=True)

    h_t = attribution_entropy(teacher_explainer, txt)
    if h_t is None:
        print("— teacher failed, skip")
        continue
    h_s = attribution_entropy(student_explainer, txt)
    if h_s is None:
        print("— student failed, skip")
        continue

    teacher_entropies.append(h_t)
    student_entropies.append(h_s)
    print(f"✓  T={h_t:.2f}  S={h_s:.2f}")

print(f"\nDone: {len(teacher_entropies)}/{len(texts)} successful\n")


# ── Stats ────────────────────────────────────────────────────────────
stat, pval = mannwhitneyu(student_entropies, teacher_entropies, alternative="less")
print(f"Teacher: {np.mean(teacher_entropies):.3f} ± {np.std(teacher_entropies):.3f}")
print(f"Student: {np.mean(student_entropies):.3f} ± {np.std(student_entropies):.3f}")
print(f"Mann-Whitney (student < teacher): U={stat:.1f}, p={pval:.4e}")


# ── Plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={"width_ratios": [1, 1.4]})
colors = ["#C44E52", "#4C72B0"]
data = [teacher_entropies, student_entropies]

ax = axes[0]
parts = ax.violinplot(data, positions=[0, 1], showmeans=True, showextrema=False)
for i, pc in enumerate(parts["bodies"]):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.35)
parts["cmeans"].set_color("black")
for i, (d, col) in enumerate(zip(data, colors)):
    jitter = rng.normal(0, 0.03, size=len(d))
    ax.scatter(np.full(len(d), i) + jitter, d, c=col, alpha=0.5, s=12, edgecolors="none")
ax.set_xticks([0, 1])
ax.set_xticklabels(["BERT Teacher\n(fine-tuned)", "TinyBERT Student\n(distilled)"], fontsize=10)
ax.set_ylabel("Attribution Entropy (bits)", fontsize=11)
ax.set_title("Attribution Entropy Distribution", fontsize=12, fontweight="bold")
ymax = max(max(teacher_entropies), max(student_entropies))
sig_text = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "n.s."))
ax.plot([0, 0, 1, 1], [ymax*1.05, ymax*1.08, ymax*1.08, ymax*1.05], c="black", lw=1)
ax.text(0.5, ymax*1.09, sig_text, ha="center", va="bottom", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax2 = axes[1]
means = [np.mean(d) for d in data]
sems = [np.std(d) / np.sqrt(len(d)) for d in data]
bars = ax2.bar(["BERT Teacher\n(fine-tuned)", "TinyBERT Student\n(distilled)"],
               means, yerr=[s * 1.96 for s in sems],
               color=colors, alpha=0.75, capsize=6, edgecolor="black", linewidth=0.5)
for bar, m in zip(bars, means):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f"{m:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax2.set_ylabel("Mean Attribution Entropy (bits)", fontsize=11)
ax2.set_title("Mean Attribution Entropy (± 95% CI)", fontsize=12, fontweight="bold")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

pval_str = f"p = {pval:.2e}" if pval < 0.001 else f"p = {pval:.4f}"
fig.suptitle(
    "Interpretability Comparison: Attribution Entropy\n"
    f"Lower = more focused attributions = more interpretable  |  Mann-Whitney {pval_str}",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig("attribution_entropy_comparison.png", dpi=200, bbox_inches="tight")
plt.savefig("attribution_entropy_comparison.pdf", bbox_inches="tight")
print("\nSaved: attribution_entropy_comparison.png / .pdf")
plt.show()

Token indices sequence length is longer than the specified maximum sequence length for this model (645 > 512). Running this sequence through the model will result in indexing errors


SANITY CHECK: verifying token counts from parquet
  [0] SPOILER | 222 tokens | "**** May contain strong spoilers ****This is a review made by StoneDraim... and ..."
  [1] SPOILER | 645 tokens | "I couldn't disagree more with the critical disappointment in the middle film of ..."
  [2] SPOILER | 163 tokens | "What a terribly depressing story about people who have no idea about happiness. ..."
  [3] SPOILER | 213 tokens | "Zathura: A Space Adventure is a 2005 action packed fantasy adventure about two b..."
  [4] SPOILER | 160 tokens | "I thought this film was good I heard a lot about the film I played the game and ..."
  Max tokens: 1243 (limit: 512)
  Total: 200 (100 spoiler, 100 no-spoiler)

Computing attributions...
  [  1/200] SPO ✓  T=6.47  S=5.97
  [  2/200] SPO   ERROR: The expanded size of the tensor (645) must match the existing size (512) at non-singleton dimension 1.  Target sizes: [1, 645].  Tensor sizes: [1, 512]
— teacher failed, skip
  [  3/200] SPO ✓  T=6.65  S=6.65
  [  

KeyboardInterrupt: 